# 🏦 Loan Approval Analysis

**Objective:** Understand what drives `loan_status` (Approved / Rejected) using exploratory data analysis, statistical testing, and predictive modeling, and identify actionable levers that improve approval likelihood.

**Dataset columns:**
`loan_id, no_of_dependents, education, self_employed, income_annum, loan_amount, loan_term, cibil_score, residential_assets_value, commercial_assets_value, luxury_assets_value, bank_asset_value, loan_status`

**Note:** The raw column names in this dataset contain leading spaces (e.g. `' education'`). Step 1 cleans this immediately so every cell after it can use normal column names.

---


## Step 0 — Setup & Imports

**What this does:** Loads all libraries needed across the notebook — pandas/numpy for data handling, matplotlib/seaborn for visualization, scipy for statistical tests, and scikit-learn for modeling.


In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats
from scipy.stats import chi2_contingency, ttest_ind

# Modeling
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Optional: install if not available in your Colab runtime
# !pip install xgboost imbalanced-learn -q
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# Plot style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

pd.set_option("display.max_columns", None)


## Step 1 — Load Data & Clean Column Names

**What this does:**
- Loads the CSV (upload it to Colab first: click the folder icon → upload, or mount Google Drive).
- Strips whitespace from column names and string values (your headers have leading spaces).
- Standardizes text case for categorical columns.

**Assumption:** The CSV structure matches the columns listed in the project brief, with `loan_status` as a clean binary Approved/Rejected outcome and no partial/pending states.


In [ ]:
# Option A: Upload directly in Colab
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(next(iter(uploaded)))

# Option B: If already in your Colab file system / Drive
df = pd.read_csv("loan_approval_dataset.csv")

# Clean column names: strip whitespace
df.columns = df.columns.str.strip()

print("Cleaned columns:")
print(df.columns.tolist())

# Clean string/object column values (strip whitespace, standardize case)
obj_cols = df.select_dtypes(include="object").columns
for col in obj_cols:
    df[col] = df[col].astype(str).str.strip()

df.head()


In [ ]:
# Basic structural check
print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate loan_id count:", df["loan_id"].duplicated().sum())


**Interpretation checklist:**
- If `Missing values` shows nonzero counts anywhere, decide whether to impute or drop before moving on (this notebook assumes a mostly-clean dataset per the stated assumptions).
- If `duplicated() loan_id` > 0, investigate — duplicate applicants could bias the analysis.


## Step 2 — Feature Engineering

**What this does:** Creates derived financial ratios that are usually more predictive of loan approval than raw values alone — because approval decisions are relative (e.g. debt relative to income), not absolute.

**New features:**
| Feature | Formula | Why it matters |
|---|---|---|
| `total_assets` | sum of all 4 asset columns | Overall collateral strength |
| `debt_to_income` | loan_amount / income_annum | How much is being borrowed relative to earnings |
| `loan_to_asset_ratio` | loan_amount / total_assets | Risk exposure relative to collateral |
| `asset_to_income` | total_assets / income_annum | Wealth cushion relative to income |


In [ ]:
df["total_assets"] = (
    df["residential_assets_value"]
    + df["commercial_assets_value"]
    + df["luxury_assets_value"]
    + df["bank_asset_value"]
)

# Avoid divide-by-zero issues: Replace zero income or asset values with NaN for ratio calculations
df["debt_to_income"] = df["loan_amount"] / df["income_annum"].replace(0, np.nan)
df["loan_to_asset_ratio"] = df["loan_amount"] / df["total_assets"].replace(0, np.nan)
df["asset_to_income"] = df["total_assets"] / df["income_annum"].replace(0, np.nan)

df[["total_assets", "debt_to_income", "loan_to_asset_ratio", "asset_to_income"]].describe()


## Step 3 — Encode Categorical Variables

**What this does:** Converts text categories into numeric form for statistical tests and modeling, while keeping the original columns for readable plots.

**Assumption:** `education` has values like `Graduate`/`Not Graduate`, `self_employed` has `Yes`/`No`, and `loan_status` has `Approved`/`Rejected`. Adjust the mapping if your actual category labels differ.


In [ ]:
print("education categories:", df["education"].unique())
print("self_employed categories:", df["self_employed"].unique())
print("loan_status categories:", df["loan_status"].unique())


In [ ]:
# Keep original Columns for plotting, add encoded versions for stats/modeling
df["education_enc"] = df["education"].map(lambda x: 1 if x.lower() == "graduate" else 0)
df["self_employed_enc"] = df["self_employed"].map(lambda x: 1 if x.lower() == "yes" else 0)
df["loan_status_enc"] = df["loan_status"].map(lambda x: 1 if x.lower() == "approved" else 0)

# Sanity check: confirm no NaNs introduced by mapping (would mean unexpected category labels)
print(df[["education_enc", "self_employed_enc", "loan_status_enc"]].isnull().sum())


## Step 4 — Exploratory Data Analysis: Univariate

**What this does:** Looks at each variable in isolation before comparing against `loan_status`, to understand scale, spread, and class balance.

**Why check class balance first:** If approvals/rejections are heavily skewed (e.g. 80/20), accuracy alone will be a misleading metric later, and we'll need to handle imbalance (Step 9).


In [ ]:
# Class balance of the target variable
plt.figure(figsize=(5,4))
sns.countplot(x="loan_status", data=df, palette="Set2")
plt.title("Loan Status Distribution")
plt.show()

print(df["loan_status"].value_counts(normalize=True) * 100)


In [ ]:
# Distributions of key numeric features
num_cols = ["cibil_score", "income_annum", "loan_amount", "loan_term", "total_assets"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color="steelblue")
    axes[i].set_title(f"Distribution of {col}")
fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()


In [ ]:
# Categorical counts
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.countplot(x="education", data=df, ax=axes[0], palette="Set3")
sns.countplot(x="self_employed", data=df, ax=axes[1], palette="Set3")
sns.countplot(x="no_of_dependents", data=df, ax=axes[2], palette="Set3")
axes[0].set_title("Education")
axes[1].set_title("Self Employed")
axes[2].set_title("No. of Dependents")
plt.tight_layout()
plt.show()


## Step 5 — Exploratory Data Analysis: Bivariate (vs `loan_status`)

**What this does:** This is the core analysis — every plot here directly compares a feature against loan outcome, to visually surface which variables separate Approved from Rejected applicants.


In [ ]:
# CIBIL score vs loan status — typically the strongest signal in these datasets
plt.figure(figsize=(6,5))
sns.boxplot(x="loan_status", y="cibil_score", data=df, palette="Set2")
plt.title("CIBIL Score by Loan Status")
plt.show()


In [ ]:
# Income, loan amount, and debt-to-income vs loan status
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.boxplot(x="loan_status", y="income_annum", data=df, ax=axes[0], palette="Set2")
sns.boxplot(x="loan_status", y="loan_amount", data=df, ax=axes[1], palette="Set2")
sns.boxplot(x="loan_status", y="debt_to_income", data=df, ax=axes[2], palette="Set2")
axes[0].set_title("Annual Income by Loan Status")
axes[1].set_title("Loan Amount by Loan Status")
axes[2].set_title("Debt-to-Income by Loan Status")
plt.tight_layout()
plt.show()


In [ ]:
# Approval rate (%) by categorical variables
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

approval_by_edu = df.groupby("education")["loan_status_enc"].mean() * 100
approval_by_edu.plot(kind="bar", ax=axes[0], color="teal")
axes[0].set_title("Approval Rate by Education")
axes[0].set_ylabel("Approval Rate (%)")

approval_by_emp = df.groupby("self_employed")["loan_status_enc"].mean() * 100
approval_by_emp.plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("Approval Rate by Self-Employment")
axes[1].set_ylabel("Approval Rate (%)")

approval_by_dep = df.groupby("no_of_dependents")["loan_status_enc"].mean() * 100
approval_by_dep.plot(kind="bar", ax=axes[2], color="slateblue")
axes[2].set_title("Approval Rate by No. of Dependents")
axes[2].set_ylabel("Approval Rate (%)")

plt.tight_layout()
plt.show()


In [ ]:
# Approval rate by loan term
plt.figure(figsize=(8,5))
term_approval = df.groupby("loan_term")["loan_status_enc"].mean() * 100
term_approval.plot(kind="bar", color="darkorange")
plt.title("Approval Rate by Loan Term")
plt.ylabel("Approval Rate (%)")
plt.xlabel("Loan Term")
plt.show()


In [ ]:
# Approval rate by total_assets quartile
df["assets_quartile"] = pd.qcut(df["total_assets"], 4, labels=["Q1 (Lowest)", "Q2", "Q3", "Q4 (Highest)"])
plt.figure(figsize=(7,5))
asset_approval = df.groupby("assets_quartile")["loan_status_enc"].mean() * 100
asset_approval.plot(kind="bar", color="seagreen")
plt.title("Approval Rate by Total Assets Quartile")
plt.ylabel("Approval Rate (%)")
plt.show()


In [ ]:
# CIBIL score bands — a common real-world cutoff analysis
bins = [0, 600, 750, 900]
labels = ["<600 (Poor)", "600-750 (Fair/Good)", "750+ (Excellent)"]
df["cibil_band"] = pd.cut(df["cibil_score"], bins=bins, labels=labels)

plt.figure(figsize=(7,5))
cibil_band_approval = df.groupby("cibil_band")["loan_status_enc"].mean() * 100
cibil_band_approval.plot(kind="bar", color="crimson")
plt.title("Approval Rate by CIBIL Score Band")
plt.ylabel("Approval Rate (%)")
plt.show()

print(cibil_band_approval)


## Step 6 — Correlation Analysis

**What this does:** Quantifies linear relationships between all numeric features (including the encoded target) in one view, to spot which variables move together with `loan_status_enc`.

**Caveat:** Correlation only captures linear relationships — a low correlation doesn't rule out a variable being important in a non-linear sense (tree-based models in Step 8 will catch that).


In [ ]:
numeric_for_corr = [
    "no_of_dependents", "education_enc", "self_employed_enc",
    "income_annum", "loan_amount", "loan_term", "cibil_score",
    "residential_assets_value", "commercial_assets_value",
    "luxury_assets_value", "bank_asset_value", "total_assets",
    "debt_to_income", "loan_to_asset_ratio", "asset_to_income",
    "loan_status_enc"
]

corr_matrix = df[numeric_for_corr].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.show()

# Correlation with the target specifically, sorted
print(corr_matrix["loan_status_enc"].sort_values(ascending=False))


## Step 7 — Statistical Significance Testing

**What this does:** Moves beyond visual patterns to formally test whether differences are statistically significant, not just noise.

- **Chi-square test** — for categorical variables (`education`, `self_employed`) vs `loan_status`. Tests whether the categories are independent of the outcome.
- **T-test** — for continuous variables (`cibil_score`, `income_annum`) vs `loan_status`. Tests whether the mean differs significantly between Approved and Rejected groups.

**Interpretation rule used throughout:** p-value < 0.05 → statistically significant relationship with `loan_status`.


In [ ]:
# Chi-square tests for categorical variables
for col in ["education", "self_employed"]:
    contingency = pd.crosstab(df[col], df["loan_status"])
    chi2, p, dof, expected = chi2_contingency(contingency)
    print(f"{col}: chi2={chi2:.3f}, p-value={p:.4f} -> "
          f"{'Significant' if p < 0.05 else 'Not significant'}")


In [ ]:
# T-tests for continuous variables
approved = df[df["loan_status_enc"] == 1]
rejected = df[df["loan_status_enc"] == 0]

for col in ["cibil_score", "income_annum", "loan_amount", "debt_to_income", "total_assets"]:
    t_stat, p_val = ttest_ind(approved[col].dropna(), rejected[col].dropna(), equal_var=False)
    print(f"{col}: t-stat={t_stat:.3f}, p-value={p_val:.4f} -> "
          f"{'Significant' if p_val < 0.05 else 'Not significant'}")


## Step 8 — Predictive Modeling

**What this does:** Builds classifiers to predict `loan_status` and, more importantly, extracts **feature importance** — a direct, ranked answer to "what drives loan approval."

**Models used:**
1. **Logistic Regression** — interpretable baseline, coefficients show direction and magnitude of effect.
2. **Decision Tree** — captures non-linear splits (e.g. CIBIL score thresholds).
3. **Random Forest** — ensemble, usually stronger and more stable feature importance.
4. **XGBoost** — gradient boosting, typically the top performer on tabular data like this.

**Assumption:** We're not doing time-based validation since there's no temporal column — a standard random train/test split is used.


In [ ]:
feature_cols = [
    "no_of_dependents", "education_enc", "self_employed_enc",
    "income_annum", "loan_amount", "loan_term", "cibil_score",
    "residential_assets_value", "commercial_assets_value",
    "luxury_assets_value", "bank_asset_value",
    "total_assets", "debt_to_income", "loan_to_asset_ratio", "asset_to_income"
]

X = df[feature_cols]
y = df["loan_status_enc"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train target balance:\n", y_train.value_counts(normalize=True))


In [ ]:
# Scale features (needed for Logistic Regression; tree models don't require it but it doesn't hurt)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

def evaluate_model(name, model, X_te, y_te):
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else None

    print(f"\n--- {name} ---")
    print("Accuracy :", round(accuracy_score(y_te, y_pred), 4))
    print("Precision:", round(precision_score(y_te, y_pred), 4))
    print("Recall   :", round(recall_score(y_te, y_pred), 4))
    print("F1 Score :", round(f1_score(y_te, y_pred), 4))
    if y_proba is not None:
        print("ROC-AUC  :", round(roc_auc_score(y_te, y_proba), 4))

    cm = confusion_matrix(y_te, y_pred)
    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Rejected","Approved"], yticklabels=["Rejected","Approved"])
    plt.title(f"Confusion Matrix - {name}")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.show()

    return y_pred, y_proba


In [ ]:
# 1. Logistic Regression
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
evaluate_model("Logistic Regression", log_reg, X_test_scaled, y_test)

# Coefficients — direction & magnitude of each feature's effect
coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": log_reg.coef_[0]
}).sort_values("coefficient", ascending=False)

plt.figure(figsize=(8,6))
sns.barplot(x="coefficient", y="feature", data=coef_df, palette="viridis")
plt.title("Logistic Regression Coefficients (Effect on Approval)")
plt.show()


In [ ]:
# 2. Decision Tree
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
evaluate_model("Decision Tree", dt, X_test, y_test)


In [ ]:
# 3. Random Forest
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
evaluate_model("Random Forest", rf, X_test, y_test)

# Feature importance
rf_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

plt.figure(figsize=(8,6))
sns.barplot(x="importance", y="feature", data=rf_importance, palette="mako")
plt.title("Random Forest — Feature Importance")
plt.show()

print(rf_importance)


In [ ]:
# 4. XGBoost
xgb = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    random_state=42, eval_metric="logloss"
)
xgb.fit(X_train, y_train)
evaluate_model("XGBoost", xgb, X_test, y_test)

xgb_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": xgb.feature_importances_
}).sort_values("importance", ascending=False)

plt.figure(figsize=(8,6))
sns.barplot(x="importance", y="feature", data=xgb_importance, palette="rocket")
plt.title("XGBoost — Feature Importance")
plt.show()


In [ ]:
# ROC curve comparison across all models
plt.figure(figsize=(7,6))
for name, model, X_te in [
    ("Logistic Regression", log_reg, X_test_scaled),
    ("Decision Tree", dt, X_test),
    ("Random Forest", rf, X_test),
    ("XGBoost", xgb, X_test),
]:
    y_proba = model.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0,1], [0,1], "k--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()


## Step 9 — Handling Class Imbalance & Hyperparameter Tuning

**What this does:**
- If Step 4 showed `loan_status` is imbalanced, this applies **SMOTE** (Synthetic Minority Over-sampling) to the training data only (never to test data — that would leak synthetic signal into evaluation).
- Runs **GridSearchCV** on Random Forest to tune hyperparameters for better generalization.

**Why train-only:** Oversampling before the split would let synthetic copies of the same applicant appear in both train and test sets, inflating test scores artificially.


In [ ]:
# Apply SMOTE only if there's meaningful imbalance (e.g. minority class < 40%)
minority_ratio = y_train.value_counts(normalize=True).min()
print("Minority class ratio in training data:", round(minority_ratio, 3))

if minority_ratio < 0.4:
    smote = SMOTE(random_state=42)
    X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
    print("Balanced training class distribution:\n", y_train_bal.value_counts())
else:
    X_train_bal, y_train_bal = X_train, y_train
    print("Classes are reasonably balanced — SMOTE not applied.")


In [ ]:
# Hyperparameter tuning for Random Forest
param_grid = {
    "n_estimators": [200, 300, 400],
    "max_depth": [5, 8, 12, None],
    "min_samples_split": [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)
grid_search.fit(X_train_bal, y_train_bal)

print("Best parameters:", grid_search.best_params_)
print("Best CV ROC-AUC:", round(grid_search.best_score_, 4))

best_rf = grid_search.best_estimator_
evaluate_model("Tuned Random Forest (Balanced)", best_rf, X_test, y_test)


## Step 10 — Summary of Findings & Recommendations

**What this does:** Consolidates everything above into a single business-readable summary — the "so what" of the analysis.

Run the cell below to auto-generate a data-driven summary table. Then use the markdown cell underneath as a template to write your conclusions once you've seen your actual output.


In [ ]:
summary_table = rf_importance.merge(
    xgb_importance, on="feature", suffixes=("_rf", "_xgb")
)
summary_table["avg_importance"] = summary_table[["importance_rf", "importance_xgb"]].mean(axis=1)
summary_table = summary_table.sort_values("avg_importance", ascending=False)

print("Top drivers of loan_status (averaged across RF & XGBoost):")
summary_table[["feature", "avg_importance"]]


### 📌 Report Template (fill in with your actual numbers once you run the notebook)

**Key drivers of `loan_status`:**
1. `cibil_score` — [state observed relationship, e.g. "applicants below X score are rejected in Y% of cases"]
2. `debt_to_income` / `loan_amount` — [state pattern]
3. `total_assets` / `loan_to_asset_ratio` — [state pattern]
4. `education`, `self_employed`, `no_of_dependents` — [state whether these were statistically significant from Step 7, or mostly noise]

**Model performance:** Best model was **[X]** with ROC-AUC of **[Y]**, meaning it separates approved vs rejected applicants [well/moderately/poorly].

---

### Assumptions Made in This Analysis
1. `cibil_score` follows the standard 300–900 Indian credit score range.
2. All monetary columns (`income_annum`, `loan_amount`, asset values) are in the same currency/unit.
3. `loan_status` is a clean binary outcome (Approved/Rejected) with no partial/pending states.
4. Each row represents an independent applicant — no relationship or duplication across rows.
5. Self-reported asset values are assumed accurate (no external verification).
6. Missing values, if present, are assumed missing-at-random unless a pattern was found in Step 1.
7. No temporal dimension exists — `loan_id` order does not imply chronology, so trend-over-time analysis isn't possible.
8. The dataset is assumed representative of the broader applicant population (not biased toward one lender's unusually strict/lenient policy).
9. A standard random train/test split was used since there is no time-based ordering to respect.

---

### Recommendations

**A. To improve model performance (predicting approval accurately):**
- Address class imbalance via SMOTE or class-weighting (done in Step 9) if approvals/rejections are skewed.
- Continue hyperparameter tuning (GridSearchCV) — Step 9 covers Random Forest; extend this to XGBoost for further gains.
- Consider ensembling/stacking top models (Random Forest + XGBoost + Logistic Regression) for better generalization.
- Collect more data or additional features (e.g. credit history length, existing liabilities) if performance plateaus.

**B. Business-side insight (what raises an applicant's actual approval odds):**
- Improving `cibil_score` is typically the single biggest lever — target score bands identified in Step 5.
- Lowering `debt_to_income` ratio (borrowing less relative to income, or demonstrating higher income).
- Increasing `total_assets` relative to the loan amount reduces perceived risk.
- Loan term effects should be read from the Step 5 chart rather than assumed — test rather than guess.
